# 2.1 Travel Agent — HTTP MCP Server

## What you will learn in this notebook

- How to connect to a **hosted (remote) MCP server** over HTTP instead of stdio
- How a real-world **travel booking API** can be exposed as MCP tools
- How combining **MCP + memory** enables a conversational travel agent
- The difference between `stdio` and `streamable_http` transports

---

### stdio vs streamable_http

| | `stdio` transport | `streamable_http` transport |
|---|---|---|
| **Server location** | Your machine (subprocess) | Remote server (cloud/internet) |
| **How it starts** | LangChain spawns the process | Server is already running |
| **Connection** | stdin/stdout pipes | HTTP requests to a URL |
| **Latency** | Near-zero (local) | Network round-trip |
| **Use case** | Your own tools, `uvx` packages | Third-party SaaS APIs, hosted services |

### The Kiwi.com MCP Server

Kiwi.com is a flight search platform. They host an MCP server at `https://mcp.kiwi.com` that exposes their flight search API as MCP tools. We connect to it here — no API key required for basic usage.

This is the future of API integrations: instead of reading API docs, writing HTTP clients, and parsing response formats, you just point your agent at an MCP endpoint and the tools are ready to use.

In [4]:
# ============================================================
# CELL 1: Load Environment Variables
# ============================================================

from dotenv import load_dotenv

load_dotenv()

True

In [5]:
# ============================================================
# CELL 2: Connect to the Kiwi.com MCP Server (HTTP Transport)
# ============================================================
# Key difference from 2.1_mcp.ipynb:
#   "transport": "streamable_http"  → connect over HTTP, not stdin/stdout
#   "url": "https://mcp.kiwi.com"  → the remote server address
#
# There is no "command" or "args" field because we don't launch
# a subprocess — the server is already running in Kiwi's cloud.
#
# LangChain sends MCP protocol messages as HTTP requests to this URL.
# The server responds with its tool definitions, and LangChain wraps
# them into standard tool objects — exactly the same interface we've
# used throughout this course, just sourced remotely.
#
# await is required because connecting and fetching tool schemas
# involves async network I/O.

from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {                         # Nickname for this server
            "transport": "streamable_http",        # HTTP transport (not stdio)
            "url": "https://mcp.kiwi.com"          # Remote MCP server URL
        },
        "tavily_mcp":{
            "transport": "http",
            "url":"https://mcp.tavily.com/mcp/?tavilyApiKey=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"
        }
    }
)

# Fetch the flight search tools from Kiwi's server
tools = await client.get_tools()

In [2]:
print(tools)

[StructuredTool(name='search-flight', description='# Search for flights\n\nSearches Kiwi.com for available flights between two locations for the given dates and passengers. City or airport names are resolved automatically, so call this whenever the user wants to search for flights — whether they gave IATA codes or just place names.\n\n## Result shape\n\nReturns `{ query, currency, passengers, resultsCount, itineraries, searchTimeMs }`. Each item in `itineraries` has:\n- `price` (number) and `priceFormatted` (e.g. "123 EUR")\n- `totalDurationSeconds`\n- `bookingUrl` — the link to book the flight\n- `outbound` (and `inbound` for return flights), each a leg with: `route` (list of airport codes including layovers, e.g. ["PRG","MAD","BCN"]), `departureTime` / `arrivalTime` (local ISO timestamps), `durationSeconds`, `stops`, `cabinClass`, and a `segments` list.\n\n## How to display\n\nRender a markdown table. Group results into cheapest (lowest `price`), shortest (lowest `totalDurationSecond

In [6]:
# ============================================================
# CELL 3: Create the Travel Agent
# ============================================================
# We combine three features for a useful travel assistant:
#
# tools=tools
#   → Kiwi.com flight search tools from the MCP server
#     The agent can search for real flights, prices, routes, etc.
#
# checkpointer=InMemorySaver()
#   → Multi-turn memory — user can refine the search without
#     repeating origin, destination, and date each time.
#     Follow-up: "Show me only direct flights" will work because
#     the agent remembers the original query.
#
# system_prompt="...No follow up questions."
#   → Key instruction: the agent should make a decision and return
#     results immediately rather than asking clarifying questions.
#     This creates a proactive assistant, not a passive one.
#     For a travel agent, users want results now — they can refine later.

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "gpt-5-nano",
    tools=tools,                    # Kiwi.com flight search tools
    checkpointer=InMemorySaver(),   # Remember the conversation
    system_prompt="You are a travel agent. No follow up questions."  # Proactive style
)

In [8]:
# ============================================================
# CELL 4: Search for a Flight
# ============================================================
# The agent will:
#   1. Parse the user's request (SFO → TYO, March 31, direct only)
#   2. Call a Kiwi.com search tool with those parameters
#   3. Parse the flight results
#   4. Present the best options in natural language
#
# The word "direct" is key — the agent should filter or sort results
# to show only non-stop flights. Watch the tool_calls in the trace
# to see exactly what parameters it sends to Kiwi.
#
# Using ainvoke() because MCP tool calls are async (HTTP requests
# to the remote Kiwi server happen on each tool call).

from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}  # Session for this user

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Get the current weather report for Hyderabad. If the current temperature is above 20°C, then search for available flights from Hyderabad (HYD) to San Francisco (SFO).")]},
    config
)

In [6]:
# ============================================================
# CELL 5: Inspect the Full Message Trace
# ============================================================
# For a flight search with one tool call, expect:
#   messages[0]  HumanMessage  → user's flight request
#   messages[1]  AIMessage     → tool call decision (flight search params)
#   messages[2]  ToolMessage   → raw flight results from Kiwi's API
#   messages[3]  AIMessage     → formatted flight options for the user
#
# The ToolMessage content (messages[2]) is worth inspecting —
# it shows the raw JSON from Kiwi's API, including prices, durations,
# airline codes, and booking links.

from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Get me a direct flight from San Francisco to Tokyo on March 31st', additional_kwargs={}, response_metadata={}, id='9db87332-0296-49ec-bd38-408bd2ca3407'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 770, 'prompt_tokens': 2632, 'total_tokens': 3402, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 704, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DN7cJoBIKiEYlQVJQ8QHnUgOTDPqI', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d22ac-0a52-7ac1-89a5-f15545cd8e6e-0', tool_calls=[{'name': 'search-flight', 'args': {'flyFrom': 'San Francisco', 'flyTo': 'Tokyo', 'departureDate': '31/03/2026', 'passengers': {'adults': 

In [7]:
# ============================================================
# CELL 6: Print Just the Final Answer
# ============================================================
# The agent's flight recommendations in natural language.
# Try adding a follow-up call using the same config:
#
#   response2 = await agent.ainvoke(
#       {"messages": [HumanMessage(content="What's the cheapest option?")]},
#       config  # Same thread_id — agent remembers the search results
#   )
#
# The memory means you can refine, compare, and drill into details
# without repeating the original destination and date.

print(response["messages"][-1].content)

Here are direct flights from San Francisco (SFO) to Tokyo (NRT) on March 31, 2026.

Cheapest options
| Route | Schedule (Local) | Cabin | Return Route | Return Schedule | Return Cabin | Price | Book |
|---|---|---|---|---|---|---:|---|
| SFO → NRT | 31/03 16:45 → 01/04 20:00 (11h 15m) | Economy | — | — | — | USD 789 | https://on.kiwi.com/qeWQmW |
| SFO → NRT | 31/03 18:36 → 02/04 21:00 (34h 24m) | Economy | — | — | — | USD 826 | https://on.kiwi.com/otJdGt |
| SFO → NRT | 31/03 11:40 → 01/04 15:00 (11h 20m) | Economy | — | — | — | USD 1196 | https://on.kiwi.com/Spv3xa |
| SFO → NRT | 31/03 11:40 → 01/04 15:00 (11h 20m) | Economy | — | — | — | USD 1238 | https://on.kiwi.com/ZRrRVf |

Shortest duration options
| Route | Schedule (Local) | Cabin | Return Route | Return Schedule | Return Cabin | Price | Book |
|---|---|---|---|---|---|---:|---|
| SFO → NRT | 31/03 16:45 → 01/04 20:00 (11h 15m) | Economy | — | — | — | USD 789 | https://on.kiwi.com/qeWQmW |
| SFO → NRT | 31/03 11:40 → 01/04 1

---
## Summary

| Concept | Key takeaway |
|---|---|
| **`streamable_http`** | Connect to remote MCP servers via URL — no subprocess, no local files |
| **Hosted MCP servers** | Third-party APIs (Kiwi, Stripe, GitHub, etc.) expose themselves as MCP endpoints |
| **`tools = await client.get_tools()`** | Same call regardless of transport type — stdio or HTTP, same interface |
| **"No follow up questions"** | System prompt trick to make agents proactive — return results immediately |
| **Memory + MCP** | `InMemorySaver` lets users refine searches without repeating all parameters |
| **`ainvoke()`** | Required when tools involve async I/O (HTTP calls to remote servers) |

### The bigger picture

This notebook shows why MCP matters for production apps. Instead of:
1. Reading Kiwi's API documentation
2. Writing an HTTP client with auth headers
3. Parsing their JSON response format
4. Writing a `@tool` wrapper

You just point `MultiServerMCPClient` at their URL and get working tools in two lines. This pattern scales: one MCP client can connect to flights, hotels, car rental, and weather simultaneously.